# AnimationStudio — Phase 5: Music Production & ACE-Step Integration on Google Colab

This notebook verifies and runs the **Phase-5 music production system**
(`PHASE5.md` / `PHASE5_STATUS.md`) described by `scripts/generate_phase5.py`.

The Phase 5 audio bible is pure Python — **no GPU required for verification**.
But this notebook also supports **real song generation** via ACE-Step on GPU runtimes.

- **CPU runtime:** offline verification + mock generation (no audio produced).
- **GPU runtime (T4+):** real ACE-Step song generation (~4.7 GB VRAM for turbo model).
- **No Drive mount needed** — nothing here touches `catalog.db`.
- Models download to the Colab VM disk on first start, **never to Drive**.

## What gets verified

| Deliverable | Where |
| --- | --- |
| Music style guide / philosophy | `Audio/Music/MUSIC_STYLE_GUIDE.md` + `src/audio_bible/` |
| 24 song categories + tempo bands | `src/audio_bible/libraries.py` |
| Voice profiles (11 characters) | `CharacterVoices/VOICE_PROFILES.md` |
| Prompt templates + negatives | `PromptTemplates/music-prompts.md` + `src/audio_bible/prompts.py` |
| Generation pipeline (ACE-Step / Mock) | `src/music_generation/` + `scripts/generate_phase5.py` |
| Review UI music hooks | `src/review_ui/app.py` (music routes) |

## Steps

1. In Cell 1 set `REPO_URL` and optionally toggle `RUN_REAL_GENERATION`.
2. Runtime -> Run all.

In [ ]:
#@title 1. Settings

import os
import subprocess
import sys

# GitHub clone URL for this studio (push master there first).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}
BRANCH = "master"  #@param ["master", "colab-gpu"]

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"

# Cell 8: push the refreshed report back to GitHub. Off -> download a zip.
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}

# GitHub PAT (Settings -> Developer settings -> Tokens). Needs Contents:
# Read+Write. Leave empty if the repo is public and you push over HTTPS creds.
GITHUB_TOKEN = ""  #@param {type:"string"}

# --- Phase 5: music generation ---

# Toggle real generation (GPU required for ace-step). When False, the
# generate cell runs --backend mock (offline verification, no audio produced).
RUN_REAL_GENERATION = False  #@param {type:"boolean"}

# ACE-Step model tier: turbo (4.7 GB VRAM, fast) or xl (9 GB, higher quality).
# T4 free-tier runs turbo comfortably; XL needs CPU offload.
ACESTEP_MODEL = "turbo"  #@param ["turbo", "xl"]

# Push each generated song to git as soon as it finishes, so an interrupted
# session never loses the audio that already completed.  Needs SYNC_TO_GITHUB
# and (for a private repo) GITHUB_TOKEN to be set above.  Requires git to be
# configured in this environment (it is, on the cloned checkout).
SYNC_MUSIC_AFTER_EACH = True  #@param {type:"boolean"}

In [ ]:
#@title 2. Clone repo and install the studio

def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


os.chdir(WORK)
if not os.path.isdir(REPO):
    # Full clone so Cell 8 can push the refreshed report back.
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "jinja2", "aiosqlite", "python-multipart",
     "pydantic", "scikit-learn"])
print("Studio installed (branch:", BRANCH, ")")
print("CPU runtime: verification + mock mode. GPU runtime: real songs via ACE-Step.")

In [ ]:
#@title 3. Preview the 24 song categories

from src.audio_bible import AudioBible

bible = AudioBible()
categories = bible.list_song_categories()
print(f"Song categories: {len(categories)}")
for cat in categories:
    print(f"  {cat}")

In [ ]:
#@title 4. Install & start the local ACE-Step service (GPU runtime only)
#@markdown *Skip this cell on CPU runtime — run Cell 5 with mock backend instead.*
#
# ACE-Step 1.5 service bring-up.  Commands CITED from the vendor README:
# https://github.com/ACE-Step/ACE-Step-1.5
#
# VRAM guidance (vendor gpu_config.py):
#   turbo: DiT 4.7 GB + VAE 0.33 + text encoder 1.2 + CUDA ctx ~0.5 = ~6.7 GB
#          Comfortable on T4 (15 GB usable).
#   xl:    DiT 9.0 GB + same overhead = ~11 GB.  Needs CPU offload on T4.
#
# Models auto-download (~10-20 GB) to the service checkout on first start.
# They live on the Colab VM disk — NEVER Google Drive.
#
# NOTE: ACE-Step 1.5 pins `requires-python = >=3.11,<3.13`, but Colab now
# ships Python 3.13+.  We therefore ALWAYS bring up the service through `uv`,
# which provisions its own compatible isolated interpreter + venv.  Preferring
# `uv` over `pip install -r requirements.txt` avoids silent startup crashes.

import os, shutil, subprocess, sys, time, urllib.request

SERVER_LOG = "/content/acestep-server.log"


def _tail(path, max_bytes=12000):
    """Return the tail of a log file (server stdout/stderr) for diagnostics."""
    try:
        with open(path, "rb") as f:
            f.seek(0, os.SEEK_END)
            size = f.tell()
            f.seek(max(0, size - max_bytes))
            return f.read().decode("utf-8", "replace").strip()
    except Exception:
        return "<no server log yet>"


def _pop_secret(keys):
    for k in keys:
        os.environ.pop(k, None)


if not RUN_REAL_GENERATION:
    print("RUN_REAL_GENERATION is False — skipping ACE-Step service setup.")
    print("Cell 5 will run with --backend mock (offline verification).")
else:
    ACE_DIR = "/content/ACE-Step-1.5"
    PYVER = f"{sys.version_info.major}.{sys.version_info.minor}"

    if not os.path.isdir(ACE_DIR):
        run(["git", "clone",
             "https://github.com/ACE-Step/ACE-Step-1.5.git", ACE_DIR])

    # ACE-Step 1.5 needs Python 3.11/3.12.  Ensure `uv` exists; it will
    # download a compatible CPython and create an isolated `.venv`.
    if shutil.which("uv") is None:
        print(f"uv not found — installing it (pip install uv) ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"],
                       check=True)
    print(f"uv sync in {ACE_DIR} (isolated py3.11/3.12 env)...")
    subprocess.run(["uv", "sync"], cwd=ACE_DIR, check=True)
    # ACE-Step does NOT honor ACESTEP_DTYPE for the DiT compute dtype on
    # pre-Ampere GPUs (upstream issue #1055): init_service_orchestrator.py
    # hard-codes torch.float16 when CUDA lacks bf16, and fp16 overflows to NaN
    # latents on a Colab T4 (fails every song with "Generation produced NaN or
    # Inf latents"). Patch the installed source so our ACESTEP_DTYPE=float32
    # override below actually takes effect.
    _orch_path = os.path.join(
        ACE_DIR, "acestep", "core", "generation", "handler",
        "init_service_orchestrator.py",
    )
    if os.path.exists(_orch_path):
        with open(_orch_path, "r", encoding="utf-8") as _fh:
            _src = _fh.read()
        _marker = "# _ACESTEP_DTYPE_OVERRIDE_PATCHED"
        if _marker not in _src:
            _old = (
                "                else:\n"
                "                    self.dtype = torch.float16"
            )
            _new = (
                "                else:\n"
                "                    self.dtype = (\n"
                "                        torch.float32\n"
                "                        if os.environ.get(\"ACESTEP_DTYPE\", \"\")\n"
                "                        .strip().lower() == \"float32\"\n"
                "                        else torch.float16\n"
                "                    )\n"
                "                    " + _marker + "\n"
            )
            if _old in _src:
                with open(_orch_path, "w", encoding="utf-8") as _fh:
                    _fh.write(_src.replace(_old, _new, 1))
                print("Patched ACE-Step init_service_orchestrator.py to honor ACESTEP_DTYPE=float32.")
            else:
                print("WARN: ACE-Step dtype branch not found; ACESTEP_DTYPE may be ignored.")
        else:
            print("ACE-Step dtype override patch already applied.")
    else:
        print("WARN: ACE-Step orchestrator not found; ACESTEP_DTYPE may be ignored.")
    ace_cmd = ["uv", "run", "acestep-api"]
    ace_cmd = ["uv", "run", "acestep-api"]
    print(f"Colab Python {PYVER}; ACE-Step runs in its own uv-provided env.")

    os.environ["ACESTEP_API_KEY"] = "colab-local-key"
    os.environ["ACESTEP_BASE_URL"] = "http://127.0.0.1:8001"

    # Default ACESTEP_NO_INIT=true => models lazy-load on first request, so the
    # port binds quickly and /health answers immediately (healthy fast).
    #
    # matplotlib backend: the Colab kernel sets MPLBACKEND to
    # module://matplotlib_inline.backend_inline, which this isolated venv does
    # not ship and is invalid headless — that crashes the server at import.
    # Force the headless Agg backend for the spawned process.
    server_env = os.environ.copy()
    server_env["MPLBACKEND"] = "Agg"
    # fp16 diffusion overflows to NaN on pre-Ampere Colab GPUs
    # (Turing/T4); every infected latent set fails with
    # "Generation produced NaN or Inf latents" (server msg #5).
    # Force float32 so generation produces valid audio.
    server_env["ACESTEP_DTYPE"] = "float32"
    with open(SERVER_LOG, "wb") as lf:
        server = subprocess.Popen(ace_cmd, cwd=ACE_DIR,
                                  stdout=lf, stderr=subprocess.STDOUT,
                                  env=server_env)

    print("Waiting for ACE-Step service to become healthy...")
    healthy = False
    for i in range(120):  # up to 10 minutes
        if server.poll() is not None:
            # Process died before becoming healthy — surface the real error.
            print("!! ACE-Step server process EXITED early "
                  f"(rc={server.returncode}).  Last log:")
            print("---- server log ----")
            print(_tail(SERVER_LOG))
            print("--------------------")
            _pop_secret(["ACESTEP_BASE_URL", "ACESTEP_API_KEY"])
            raise SystemExit("ACE-Step server died during startup. "
                             "See the log above.")
        try:
            urllib.request.urlopen(
                f"{os.environ['ACESTEP_BASE_URL']}/health", timeout=2)
            print(f"ACE-Step healthy after {i*5}s")
            healthy = True
            break
        except Exception:
            time.sleep(5)

    if not healthy:
        print("!! ACE-Step did not become healthy in 10 minutes. "
              "Server still running?", server.poll() is None)
        print("---- server log (tail) ----")
        print(_tail(SERVER_LOG))
        print("---------------------------")
        _pop_secret(["ACESTEP_BASE_URL", "ACESTEP_API_KEY"])
        print("Falling back to --backend mock for generation.")


In [ ]:
#@title 5. Generate songs (or run offline verification)
#@markdown Real generation requires Cell 4 to have started the ACE-Step service.
#@markdown On CPU runtime this cell runs --backend mock (no audio produced).
#
# Captures stdout/stderr and, on a non-zero exit, prints the tail so real
# generation failures (e.g. first-run lazy model load) are visible instead of a
# bare CalledProcessError with hidden output.

import subprocess

if RUN_REAL_GENERATION and os.environ.get("ACESTEP_BASE_URL"):
    backend_flag = "ace-step"   # alias accepted by generate_phase5.py
    print("Generating songs via ACE-Step (real backend)...")
else:
    backend_flag = "mock"
    print("Generating songs via mock backend (offline verification)...")

cmd = [sys.executable, "scripts/generate_phase5.py",
     "--generate", "--backend", backend_flag]

# Per-song git sync: push each finished song + manifest right away so an
# interrupted/restarted session never loses completed audio.  Reuses the
# exact branch/token/repo/identity from Settings.  Only enabled for real
# ACE-Step generation — mock verification stays local.
if RUN_REAL_GENERATION and SYNC_MUSIC_AFTER_EACH and SYNC_TO_GITHUB:
    cmd += [
        "--git-sync-every", "1",
        "--git-branch", BRANCH,
        "--git-token", GITHUB_TOKEN,
        "--git-remote-url", REPO_URL,
        "--git-name", GIT_NAME,
        "--git-email", GIT_EMAIL,
    ]
    print("Git sync enabled: pushing each finished song immediately.")

proc = subprocess.run(
    cmd,
    capture_output=True, text=True,
)
print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
if proc.stderr:
    print("---- stderr ----")
    print(proc.stderr.rstrip())

if proc.returncode != 0:
    # Surface the last stderr lines so failures are diagnosable.
    lines = (proc.stdout + "\n" + proc.stderr).strip().splitlines()
    tail = lines[-25:]
    print("\n!! generate_phase5.py exited", proc.returncode, "(non-zero).")
    for line in tail:
        print("   |", line)
    # The ACE-Step server itself is what fails the generation jobs; its
    # own log holds the real exception (model load / CUDA / validation)
    # that the job-level "unknown error" hides. Dump the tail so it is
    # diagnosable.
    print("\n---- ACE-Step server log (tail) ----")
    print(_tail(SERVER_LOG))
    print("------------------------------------")
    raise SystemExit(
        f"generate_phase5.py exited {proc.returncode}. See output and "
        "the ACE-Step server log above."
    )


In [ ]:
#@title 6. Run the Phase-5 test suites

# tests/test_audio_bible.py        — bible libraries, briefs, prompts,
#                                    production system, doc<->code consistency.
# tests/test_music_generation.py   — music_generation backend protocol,
#                                    ACE-Step/Suno/Mock adapters.
# tests/test_generate_phase5.py    — generation mode, manifest, batch loop.
# tests/test_review_ui_music.py    — Review UI music page + routes.
run([sys.executable, "-m", "pytest",
     "tests/test_audio_bible.py",
     "tests/test_music_generation.py",
     "tests/test_generate_phase5.py",
     "tests/test_review_ui_music.py",
     "-q"])

In [ ]:
#@title 7. Review manifest and report

import json as _json
from IPython.display import Markdown, display

manifest_path = f"{REPO}/Audio/Music/manifest.json"
if os.path.exists(manifest_path):
    with open(manifest_path) as fh:
        manifest = _json.load(fh)
    songs = manifest.get("songs", [])
    print(f"Manifest: {len(songs)} song(s) recorded (version {manifest.get('version', '?')})")
    for s in songs[:5]:
        print(f"  {s.get('file', '?')}  backend={s.get('backend','?')}  seed={s.get('seed','?')}")
    if len(songs) > 5:
        print(f"  ... and {len(songs)-5} more")
else:
    print("No manifest.json yet — run Cell 5 to generate songs.")

report_path = f"{REPO}/PHASE5_REPORT.md"
if os.path.exists(report_path):
    with open(report_path) as fh:
        report = fh.read()
    display(Markdown(report[:6000]))
    print("... (full file:", len(report), "chars)")
else:
    print("No PHASE5_REPORT.md found.")

In [ ]:
#@title 8. Sync the refreshed report (GitHub push or manual download)

from datetime import datetime

if SYNC_TO_GITHUB:
    sys.path.insert(0, f"{REPO}/colab")
    from git_sync import _basic_auth_header

    def _run(cmd, **kw):
        print("+ " + " ".join(cmd))
        return subprocess.run(cmd, check=False, cwd=REPO, **kw)

    _run(["git", "config", "user.name", GIT_NAME])
    _run(["git", "config", "user.email", GIT_EMAIL])
    _run(["git", "add", "Audio/Music/", "PHASE5_REPORT.md"])
    dirty = _run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if dirty.stdout.strip():
        _run(["git", "commit", "-m",
              f"Phase 5 report + music {datetime.now():%Y-%m-%d %H:%M}"])
        if GITHUB_TOKEN:
            _run(["git", "-c",
                  f"http.extraheader=Authorization: {_basic_auth_header(GITHUB_TOKEN)}",
                  "push", "origin", BRANCH])
        else:
            _run(["git", "push", "origin", BRANCH])
    else:
        print("Report unchanged — nothing to push.")
else:
    from google.colab import files
    files.download(f"{REPO}/PHASE5_REPORT.md")
    print("Downloaded PHASE5_REPORT.md.")

## Next steps

- The refreshed `PHASE5_REPORT.md` and `Audio/Music/manifest.json` now live
  in your repo (or were downloaded).
- Browse the music bible in the Review UI: **Dashboard → Music** — 24 categories
  with tempo/key/duration standards, the **Song Prompt Builder** (`POST /music/prompt`),
  and single-song generation with background job polling.
- For real songs, re-run Cell 4 on a GPU runtime with `RUN_REAL_GENERATION = True`,
  then re-run Cell 5.  See `Audio/Music/README.md` for the live-smoke checklist.
- Re-run Cells 5–8 any time `Audio/*.md` or `src/audio_bible/` change.
- With `SYNC_MUSIC_AFTER_EACH = True`, Cell 5 pushes each finished song
  + `Audio/Music/manifest.json` to git right away (one commit per song), so
  an interrupted session never loses completed audio.  `--git-sync-every N`
  and `--timeout-s` (default 900s) control sync frequency and per-job timeout.